In [49]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": Python or Json or Regex
        "solution_criteria": "Solution criteria about what a good solution would look like"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    print(text)

    return json.loads(text)
    

In [52]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


[
    {
        "task": "Extract all S3 bucket names from an AWS CloudFormation template",
        "format": "Regex",
        "solution_criteria": "A regular expression that accurately captures S3 bucket name references in CloudFormation YAML/JSON format, including those in resource names and properties. Should match patterns like 'AWS::S3::Bucket' resource definitions and bucket property values."
    },
    {
        "task": "Parse an AWS IAM policy JSON document and return a list of all allowed actions",
        "format": "Python",
        "solution_criteria": "A Python function that takes an IAM policy JSON as input and returns a deduplicated list of all actions granted in the 'Action' fields across all statements. Should handle wildcards and multiple action formats correctly."
    },
    {
        "task": "Create a JSON template for an AWS Lambda environment variables configuration",
        "format": "Json",
        "solution_criteria": "A valid JSON object that represents AWS La

In [53]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output 

In [54]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_by_syntax(test_case, output):
    if test_case["format"] == "JSON":
        return validate_json(output)
    
    elif test_case["format"] == "Python":
        return validate_python(output)
    
    else:
        return validate_regex(output)

In [55]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria to base your grading:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [56]:
def grade_test_case(test_case):
    """Call run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_by_syntax(test_case, output)
    
    composite_score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": composite_score,
        "reasoning": reasoning
    }


In [57]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each test case"""
    results = []

    for test_case in dataset: 
        result = grade_test_case(test_case)
        results.append(result)

    mean_score = mean(result["score"] for result in results) 
    print(mean_score)


    return results

In [58]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

7.666666666666667


In [59]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "\nimport json\nimport re\nimport sys\n\ndef extract_s3_buckets(template_str):\n    \"\"\"Extract S3 bucket names from CloudFormation template\"\"\"\n    try:\n        template = json.loads(template_str)\n    except json.JSONDecodeError:\n        template = {}\n    \n    s3_buckets = set()\n    \n    # Search through Resources section\n    resources = template.get('Resources', {})\n    for resource_name, resource_config in resources.items():\n        resource_type = resource_config.get('Type', '')\n        \n        # Check if it's an S3 bucket resource\n        if resource_type == 'AWS::S3::Bucket':\n            # Use BucketName property if available\n            bucket_name = resource_config.get('Properties', {}).get('BucketName')\n            if bucket_name:\n                s3_buckets.add(bucket_name)\n            else:\n                # Otherwise use logical ID as bucket name\n                s3_buckets.add(resource_name)\n        \n        # Check Properties 